In [1]:
import os
import math
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn.functional as F  
from torch.utils.tensorboard import SummaryWriter

from Utils.CADTensorGenerator import CADTensorGenerator
from Utils.CADVisualizer   import CADVisualizer
from HDVClassNet.PP_net import PPNet
from HDVClassNet.VoronoiDecorder import VoronoiDecoder
from Training.MainTrain import TrainingConfig, NN_Trainer
from neuraltomo_fem import run_fem_loss
from problems.TipCantilever_30_20_20_midLoad import TipCantilever_30_20_20_midLoad
from problems.ThickenShell import ThickenShell

import pyvista as pv
try:
    pv.set_jupyter_backend("trame")
except Exception:
    pass

# ---- Reproducibility (recommended for D_params comparisons) ----
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
print("Code Directory:", BASE)
TesPartsDir = BASE / "Testparts" 
print("Test Step files Directory:", TesPartsDir)
# ---- Device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


Code Directory: /home/arash/VoronoiImp-main
Test Step files Directory: /home/arash/VoronoiImp-main/Testparts
device: cuda


In [ ]:
viz = CADVisualizer()
# Laoding model and extracting mesh and tensors as input
FreeFormSurf1  = TesPartsDir / "FreeFormCrv1.stp"
FreeFormSurf2A = TesPartsDir / "FreeFormSurf2A.STEP"
YachtBodypart  = TesPartsDir / "YachtBodypart.stp"
CircularSurf1  = TesPartsDir / "CircularSurf1.stp"
Cube           = TesPartsDir / "Cube.stp"
CircularSur2   = TesPartsDir / "CircularSur2.stp"
Conic          = TesPartsDir / "Conic.stp"
CircularHoles  = TesPartsDir / "CircularHoles.stp"
FullCylinder   = TesPartsDir / "FullCylinder.stp"
Sphere         = TesPartsDir / "Sphere.stp"
SphereTap      = TesPartsDir / "SphereTap.stp"
Tidebottle     = TesPartsDir / "Tidebottle.STEP"


shape_path = CircularSurf1

Case_name = shape_path.stem
print(Case_name)
generator = CADTensorGenerator(
    deflection=0.1,
    angle=0.001,
    metric_tol=1e-9,
    det_min=1e-5,
    n_u=75,
    n_v=75,
    device=device,
)

mesh_df, faces_df, tensors = generator.generate_from_file(
    shape_path=str(shape_path),
    input_ring=1,
    mode="mesh", #"1: mesh" "2:Sampled_points "
    M_per_face=2000,
    pool_size_factor=10,
    fps_pool_factor=4,
    use_fps=True,
    triangulation_max_edge_rel=0.1,
)

uv = tensors["uv"]
points_xyz = tensors["points_xyz"]
uv = tensors["uv"]
Xu = tensors["Xu"]
Xv = tensors["Xv"]
points_xyz = tensors["points_xyz"]
face_areas = tensors["face_areas"]
faces_ijk = tensors["faces_ijk"]
face_id = tensors["face_id"]
boundary_idx_ring1 = tensors["boundary_idx_ring1"]
pv_faces = tensors["pv_faces"]
bbx_all = list(tensors["BBX"].values())



xmin = min(b["xmin"] for b in bbx_all)
xmax = max(b["xmax"] for b in bbx_all)
ymin = min(b["ymin"] for b in bbx_all)
ymax = max(b["ymax"] for b in bbx_all)
zmin = min(b["zmin"] for b in bbx_all)
zmax = max(b["zmax"] for b in bbx_all)

dx = xmax - xmin
dy = ymax - ymin
dz = zmax - zmin


print(f"Number of faces: {tensors['num_faces']}")
print(f"Number of Sampled points: {uv.shape[0]}")
print(f"Global BBX dimensions: dx={dx:.4f}, dy={dy:.4f}, dz={dz:.4f}")

viz.visualize_show_Model(points_xyz, pv_faces)


pts = points_xyz.detach().cpu().numpy()
cloud = pv.PolyData(pts)
plotter = pv.Plotter()
plotter.add_mesh(cloud, render_points_as_spheres=True, point_size=6)
plotter.show()

CircularSurf1
MinVolFrac: 0.07839997112751007
Number of faces: 1
Number of Sampled points: 5776
Global BBX dimensions: dx=10.0000, dy=2.2280, dz=3.9712


Widget(value='<iframe src="http://localhost:38157/index.html?ui=P_0x7471260769e0_0&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:38157/index.html?ui=P_0x747126077a60_1&reconnect=auto" class="pyvi…

In [3]:
fixed_height_shell= 0.5
shell_problem = ThickenShell(
    thickness=fixed_height_shell,
    BC_dir = "x",
    Load_magnitude=0.0001,
    voxel_size=0.2,
    extra_layers=1,
    tensors=tensors,
    tangential_tol=0.1,
)

fem = run_fem_loss.NeuralTOMOFEM(shell_problem, device=device, isotropic=False)
shell_problem.debug_voxel_stats()
shell_problem.show_voxels_surface_and_bc()

=== Voxel Stats ===
brep_bbox: {'xmin': 0.0, 'xmax': 10.0, 'ymin': -0.12709733581261684, 'ymax': 2.1009186367176094, 'zmin': -2.032529852280162, 'zmax': 1.9387036171997998}
mesh: {'nelx': 55, 'nely': 16, 'nelz': 25, 'elemSize': array([0.2, 0.2, 0.2]), 'type': 'grid'}
elem_centers shape: (25, 55, 16, 3)
node_coords shape: (26, 56, 17, 3)
occupied voxels: 4437
total voxels: 22000
occupancy ratio: 0.20168181818181818
voxelized volume: 35.49600000000001
thickness: 0.5
voxel_size: 0.2
target approx volume (sum(face_areas)*thickness): 32.80028533935547
volume ratio voxel/target: 1.082185707616698


Widget(value='<iframe src="http://localhost:38157/index.html?ui=P_0x7471260146a0_2&reconnect=auto" class="pyvi…

In [5]:
cfg = TrainingConfig(
    seed_number=10,
    use_anisotropy=False,
    fixed_height=fixed_height_shell,
    target_volfrac=0.3,
    seed_repulsion_sigma=0.08,
    boundary_margin=0.05,
    w_min=0.0005,
    w_max=0.5,

    lam_fem=0.5,
    lam_vol=20.0,
    lam_rep=0.015,
    lam_bnd=0.002,

    lam_strut=0.02,
    lam_strut_edge=1.0,
    lam_strut_void=0.2,

    lr_seed_refine=0.003,
    lr_delta_head=1e-4,
    lr_mlp=2e-4,
    lr_w_head=2e-3,
    lr_h_head=2e-4,

    comp_normalize_by=None,
    normalize_losses=False,

    fem_density_floor=0.02,
    skip_bad_fem_steps=True,

    num_steps=2000,
    context_vector_size=8,

    tau=0.1,
    beta=0.02,

    log_every=100,
    early_stop_start=500,
    patience=500,
    min_delta=1e-4,

    experiment_name=str(Case_name),
    tensorboard_enabled=True,
    tb_log_histograms_every=100,

    # new fields for the rewritten trainer
    scheduler_milestones=(500, 1000),
    scheduler_gamma=0.2,
    Offset_scale=0.5,
    save_fem_debug_history=True,
    grad_clip_norm=1.0,
)

trainer = NN_Trainer(
    generator=generator,
    viz=viz,
    decoder_cls=VoronoiDecoder,
    ppnet_cls=PPNet,
    fem=fem,
    shell_problem=shell_problem,
    config=cfg,
)

result = trainer.train(
    face_tensors=tensors["face_tensors"],
)

trainer.visualize_result_stepwise(result, points_xyz, faces_ijk)
trainer.visualize_result_final(result, points_xyz, faces_ijk, thr=0.5, show_solid=False)

TensorBoard log dir: runs/CircularSurf1
New best_step=0 | best_score=0.090944 | vol_eff=0.238481 | comp=2.658407e-04 | w=2.500662e-01
[00000] L_total=9.0944e-02 | L_vol=3.785e-03 L_fem=2.658e-04 L_strut=7.070e-01 L_rep=1.143e-08 L_bnd=4.889e-01 | vol=0.255 vol_eff=0.238 (/0.300) comp=2.658e-04 w=2.501e-01 | Lse=7.070e-01 Lsv=0.000e+00 rho(min/mean/max)=0.001/0.259/0.902 | rho_b=0.185 rho_v=0.203 | Δrho=0.00e+00 Δseed=0.00e+00 grad_mean=9.39e-05 | fem=OK | best=9.0944e-02@0
[00100] L_total=8.0445e-02 | L_vol=3.266e-03 L_fem=2.597e-04 L_strut=7.021e-01 L_rep=6.719e-08 L_bnd=4.811e-01 | vol=0.259 vol_eff=0.243 (/0.300) comp=2.597e-04 w=3.899e-01 | Lse=7.021e-01 Lsv=0.000e+00 rho(min/mean/max)=0.001/0.263/0.891 | rho_b=0.185 rho_v=0.206 | Δrho=2.86e-02 Δseed=6.12e-03 grad_mean=1.37e-05 | fem=OK | best=8.0492e-02@96
[00200] L_total=8.0089e-02 | L_vol=3.248e-03 L_fem=2.597e-04 L_strut=7.019e-01 L_rep=7.278e-08 L_bnd=4.814e-01 | vol=0.259 vol_eff=0.243 (/0.300) comp=2.597e-04 w=4.482e-01 | Ls

Widget(value='<iframe src="http://localhost:38157/index.html?ui=P_0x746e9961ba90_5&reconnect=auto" class="pyvi…

threshold=0.5 (manual) | solid%=24.948%


Widget(value='<iframe src="http://localhost:38157/index.html?ui=P_0x746e744f3c10_6&reconnect=auto" class="pyvi…

(UnstructuredGrid (0x746e74481240)
   N Cells:    3439
   N Points:   2055
   X Bounds:   0.000e+00, 1.000e+01
   Y Bounds:   -1.271e-01, 2.101e+00
   Z Bounds:   -2.033e+00, 1.939e+00
   N Arrays:   3,
 0.5)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
for t in [0.4,0.5, 0.6, 0.7]:
    trainer.visualize_result_final(result, points_xyz, faces_ijk, thr=t, show_solid=False)